# Hudson Link: is the gtfs.zone feed live?

Pulls `https://rt.gtfs.zone/hudsonlink/vehicle_positions.pb` through `shared.gtfs_rt` (agency `hudsonlink` in `shared/agencies.yaml`), checks how fresh it is, caches the snapshot and maps the vehicles. Same playbook as `analyses/runbooks/inspect_gtfs_rt/`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import folium
import pandas as pd
import yaml

from shared.gtfs_rt import get_single_pull


def load_carto_api_key():
    """Walk up from the notebook to the repo's gitignored config_secrets.yml."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        path = parent / "config_secrets.yml"
        if path.exists():
            key = (yaml.safe_load(path.read_text()) or {}).get("api_keys", {}).get("carto")
            if not key:
                print("Warning: no CARTO api key set in config_secrets.yml; map tiles may be rate-limited.")
            return key
    print("Warning: config_secrets.yml not found; CARTO map tiles may be rate-limited.")
    return None


carto_api_key = load_carto_api_key()

## Pull the feed

In [ ]:
df = get_single_pull("hudsonlink", "vehicle_positions")
print(f"{len(df)} vehicles")
df

## Freshness

Age of each vehicle's last fix against now. Under a few minutes means the scraper is live.

In [ ]:
now = pd.Timestamp.now(tz="UTC")
df["fix_time"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
df["age_s"] = (now - df["fix_time"]).dt.total_seconds().round()
print(f"Pulled {now:%Y-%m-%d %H:%M:%S} UTC; newest fix {df['age_s'].min():.0f} s old, oldest {df['age_s'].max():.0f} s old")
df[["vehicle_id", "trip_id", "route_id", "fix_time", "age_s"]]

## Cache the raw snapshot

In [ ]:
pulled_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
raw_dir = Path(f"data/raw/vehicle_positions/{pulled_at}")
raw_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(raw_dir / "data.parquet", index=False)

## Map every vehicle

In [ ]:
tiles = "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png"
if carto_api_key:
    tiles += f"?key={carto_api_key}"
m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=11, tiles=tiles, attr="CARTO")
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=5, color="#2A6F97", fill=True, fill_opacity=0.85,
        popup=f"{row['route_id']} \u00b7 vehicle {row['vehicle_id']}",
    ).add_to(m)

output_dir = Path(f"outputs/{datetime.now(timezone.utc).date()}/charts")
output_dir.mkdir(parents=True, exist_ok=True)
m.save(output_dir / "vehicle_positions_map.html")
m

*Source: gtfs.zone GTFS-Realtime (Hudson Link)*